# 3 · GRPO (Group Relative Policy Optimization)
The model learns by **trying several answers and repeating what scores well**, like practicing basketball shots.

## When is GRPO worth it?
| Prompting is enough | GRPO helps |
|---|---|
| the model already answers well | the model is sometimes right, sometimes wrong |
| answers cannot be checked by code | code can check an answer (a command, a math result, a test) |

## The task: the same shop search box as notebook 1
`black sneakers, under 50 dollars, newest first` → `FIND shoes color=BK max=50 sort=new`

The model keeps the long prompt with every rule. A **score function** grades each try:
- **A. Prompting only** (the same as notebook 1).
- **B. The same prompt + GRPO.**

## Key words
| Word | Meaning |
|---|---|
| **try** | one answer written by the model, with a little randomness, so each try can differ |
| **group** | several tries for the same request (here 4) |
| **score** | how good a try is, from 0 to 1, given by our score function |
| **advantage** | a try's score minus the group average. Above average → do more of it. |
| **epoch** | one pass through all training requests |

## Step 1 · Load the model
A small open model (0.8 billion parameters). It downloads once.

In [ ]:
import random, warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

warnings.filterwarnings('ignore')             # keep the output readable
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

## Step 2 · Create 300 examples
The same 300 examples as notebook 1 (same rules, same random seed).
Training uses only the requests. The correct command is used only by the score function.

In [ ]:
ITEMS = {'shirt': ['shirt', 't-shirt', 'tee'],
         'shoes': ['shoes', 'sneakers', 'trainers'],
         'jacket': ['jacket', 'coat'],
         'hoodie': ['hoodie', 'sweatshirt']}
COLORS = {'red': 'RD', 'blue': 'BL', 'black': 'BK', 'white': 'WH'}
PRICES = ['under {} dollars', 'less than ${}', 'below {}', 'max {} bucks']
SORTS = {'price_asc': ['cheapest first', 'lowest price first'],
         'new': ['newest first', 'latest arrivals']}
STARTS = ['', 'show me ', 'I want ', 'looking for ']

def make_example():
    item = random.choice(list(ITEMS))
    color = random.choice(list(COLORS))
    request = random.choice(STARTS) + color + ' ' + random.choice(ITEMS[item])
    command = f'FIND {item} color={COLORS[color]}'
    if random.random() < 0.6:                       # sometimes a price limit
        price = random.choice([20, 30, 50, 80, 100])
        request += ', ' + random.choice(PRICES).format(price)
        command += f' max={price}'
    if random.random() < 0.5:                       # sometimes a sort order
        sort = random.choice(list(SORTS))
        request += ', ' + random.choice(SORTS[sort])
        command += f' sort={sort}'
    return request, command

random.seed(0)
examples = {}
while len(examples) < 300:
    request, command = make_example()
    examples[request] = command                     # each request only once
examples = list(examples.items())
TRAIN, TEST = examples[:260], examples[260:]

for request, command in TRAIN[:4]:
    print(f'{request:50} -> {command}')

## Step 3 · A. Prompting only
Every rule in the prompt, no training. The same as notebook 1.

In [ ]:
RULES = '''Convert a shopping request into a search command.
Format: FIND <item> color=<code> max=<price> sort=<order>   (max and sort only if asked)
Items: shirt = shirt, t-shirt, tee | shoes = shoes, sneakers, trainers | jacket = jacket, coat | hoodie = hoodie, sweatshirt
Colors: red=RD, blue=BL, black=BK, white=WH
Sort: price_asc = cheapest first, lowest price first | new = newest first, latest arrivals
Examples:
red tee, below 30 -> FIND shirt color=RD max=30
I want white coat, newest first -> FIND jacket color=WH sort=new
looking for black trainers -> FIND shoes color=BK
Reply with only the command.'''

def long_prompt(request):
    return f'{RULES}\n\nRequest: {request}'

def ask(model, prompt, randomness=False):
    x = tok.apply_chat_template([{'role': 'user', 'content': prompt}], add_generation_prompt=True,
                                return_tensors='pt', return_dict=True).to('cuda')
    y = model.generate(**x, max_new_tokens=30, do_sample=randomness)
    return tok.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, make_prompt, show=3):
    model.eval()
    mistakes = []
    for request, command in TEST:
        answer = ask(model, make_prompt(request))
        if answer != command:
            mistakes.append((request, command, answer))
    for request, command, answer in mistakes[:show]:
        print(f'  {request}\n     expected: {command}\n     got:      {answer!r}')
    accuracy = 1 - len(mistakes) / len(TEST)
    print(f'accuracy: {accuracy:.0%}  ({len(TEST) - len(mistakes)} of {len(TEST)} correct)')
    return accuracy

prompt_only = evaluate(model, long_prompt)

## Step 4 · The score function (the heart of GRPO)
- **1.0** exactly the correct command
- **0 to 0.8** partly right: the share of command parts found in the answer

Partial scores matter: they show which tries are *closer* to the goal.

In [ ]:
def score(answer, command):
    if answer.strip() == command:
        return 1.0                                  # exactly right
    parts = command.split()                         # e.g. FIND, shoes, color=BK, max=50
    found = [p for p in parts if p in answer.split()]
    return 0.8 * len(found) / len(parts)            # partly right

c = 'FIND shoes color=BK max=50 sort=new'
for answer in [c, 'FIND shoes color=black max=50 sort=new', 'Here is the command: ' + c, 'black sneakers']:
    print(f'{score(answer, c):.2f}   {answer}')

## Step 5 · How GRPO learns: one real group
The model makes 4 tries for the same request. Each try gets a score and an advantage.
Training makes the above-average tries more likely, and the below-average ones less likely.
If all 4 scores are equal, there is nothing to learn. Run the cell again for a new group.

In [ ]:
request, command = TRAIN[0]
tries = [ask(model, long_prompt(request), randomness=True) for _ in range(4)]
scores = [score(t, command) for t in tries]
average = sum(scores) / len(scores)

print(request, '->', command, '\n')
for t, s in zip(tries, scores):
    print(f'score {s:.2f}   advantage {s - average:+.2f}   {t!r}')

## Step 6 · Prepare the training
The trainer calls `reward()` with a batch of tries plus the matching `command` column.
`num_generations=4` means 4 tries per request.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

def reward(completions, command, **kwargs):
    return [score(c[0]['content'], cmd) for c, cmd in zip(completions, command)]

data = Dataset.from_list([
    {'prompt': [{'role': 'user', 'content': long_prompt(r)}], 'command': c} for r, c in TRAIN])

trainer = GRPOTrainer(
    model=model,
    processing_class=tok,
    train_dataset=data,
    reward_funcs=reward,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=GRPOConfig(num_generations=4, max_completion_length=32,
                    num_train_epochs=1, learning_rate=1e-4, logging_steps=10),   # 1 epoch per call
)
trainer.model.print_trainable_parameters()

## Step 7 · Train, one epoch at a time
Each step: 2 requests × 4 tries → scores → advantages → adjust the add-on.
The slowest notebook: the model writes every try before it can learn.

**Look for:** the `reward` column (average score) going up, and the test accuracy going up.

In [ ]:
for epoch in [1, 2]:
    trainer.train()                                 # one pass over the 260 requests
    print(f'\nafter epoch {epoch}:')
    with_grpo = evaluate(trainer.model, long_prompt)

## Step 8 · Did it work?
It worked if the `reward` went up during training and B clearly beats A.

In [ ]:
print(f'A. prompting only: {prompt_only:.0%} correct')
print(f'B. prompt + GRPO:  {with_grpo:.0%} correct')

if prompt_only >= 0.95:
    print('\nPrompting alone was already enough: no training needed.')
elif with_grpo - prompt_only >= 0.10:
    print('\nIt worked: GRPO clearly beat prompting only.')
else:
    print('\nNo clear gain: try another epoch, or check that the tries in Step 5 get different scores.')

## Summary
- **Needs:** requests + a score function (no written answers to copy)
- **Learns:** to repeat the tries that score above average
- **Signs it works:** the `reward` goes up, the test accuracy goes up
- **Keep in mind:** GRPO can only reinforce what the model already tries sometimes, so it works best when the model is already partly right